In [14]:
import requests
import json
import os
import sys
import pandas as pd
import geopandas as gpd
import time
from IPython.display import Markdown
from shapely import wkb
import glob

In [2]:
import sys
sys.path.append(r"C:\Users\eliya\Documents\עבודה\עוזר מחקר\4. פרויקטים\1. feature completness\OSM-Feature-Completeness")

### geoparquet extraction

In [7]:
# Load the file
gdf = pd.read_parquet(r"H:\.shortcut-targets-by-id\1vC82Zl3hhtFy63TpICgdDiHqTHm5dv0h\OSM Projects\Data\overture_map_results\region_area-2025-10-22.0_dataset_counts_strtree.parquet")

In [10]:
compact_regions = gdf[['geometry', 'bbox', 'country', 'region', 'openstreetmap', 'esri_community_maps', 'instituto_geografico_nacional_espana',
                     'google_open_buildings', 'microsoft_ml_buildings', 'doi_10_5281_zenodo_8174931', 'usgs_lidar']].copy()

compact_regions['geometry'] = compact_regions['geometry'].apply(wkb.loads) # Fixing geometry column

In [11]:
compact_regions.head()

,geometry,bbox,country,region,openstreetmap,esri_community_maps,instituto_geografico_nacional_espana,google_open_buildings,microsoft_ml_buildings,doi_10_5281_zenodo_8174931,usgs_lidar
0,"POLYGON ((12.5155494 43.9398893, 12.516219 43....","{'xmax': 12.516959190368652, 'xmin': 12.469887...",SM,SM-04,387,0,0,0,96,0,0
1,"POLYGON ((28.8541043 47.8171267, 28.8537056 47...","{'xmax': 29.028268814086914, 'xmin': 28.595144...",MD,MD-RE,6246,0,0,0,26544,0,0
2,"POLYGON ((14.7043091 46.2869076, 14.7047036 46...","{'xmax': 14.878803253173828, 'xmin': 14.695601...",SI,SI-030,2365,0,0,0,201,0,0
3,"POLYGON ((-70.17954450000001 -4.0214807, -70.1...","{'xmax': -69.94135284423828, 'xmin': -77.82595...",PE,PE-LOR,177349,0,0,136045,105941,0,0
4,"POLYGON ((-6.3790366 23.1076079, -7.9877222 19...","{'xmax': -5.333332538604736, 'xmin': -9.149688...",MR,MR-01,58126,0,0,186933,107527,0,0


In [ ]:
# Filter for island countries
regions = ['PT', 'IE', 'IL', 'SO', 'UY', 'EC', 'NP', 'KR']
test_data_overture = compact_regions[compact_regions['country'].isin(regions)].copy()

### Extracted Ohsome data

In [ ]:
# path files
files = glob.glob(r"C:\Users\eliya\Documents\עבודה\עוזר מחקר\4. פרויקטים\1. feature completness\OSM-Feature-Completeness\1) Data\4) overture comparisson\geometry_clipped_data\*.csv")

In [ ]:
bld_counts = []
bld_areas = []

for f in files:
    name = os.path.basename(f)

    if "bld_counts_by_region" in name:
        bld_counts.append(pd.read_csv(f))

    elif "bld_areas_by_region" in name:
        bld_areas.append(pd.read_csv(f))

df_counts = pd.concat(bld_counts, ignore_index=True)
df_areas = pd.concat(bld_areas, ignore_index=True)


In [19]:
df_counts

,timestamp,value,ISO,region
0,2008-01-22T00:00:00Z,0.0,EC,EC-S
1,2008-02-22T00:00:00Z,0.0,EC,EC-S
2,2008-03-22T00:00:00Z,0.0,EC,EC-S
3,2008-04-22T00:00:00Z,0.0,EC,EC-S
4,2008-05-22T00:00:00Z,0.0,EC,EC-S
...,...,...,...,...
29313,2025-06-22T00:00:00Z,4382.0,UY,UY-CA
29314,2025-07-22T00:00:00Z,4606.0,UY,UY-CA
29315,2025-08-22T00:00:00Z,4606.0,UY,UY-CA
29316,2025-09-22T00:00:00Z,4605.0,UY,UY-CA


### Applying function to check completeness

In [ ]:
# Updating function to return an object with data

In [38]:
# Assess the feature completness measure of some polygon using cumulative feature counts and lengths/areas
def assess_feature_completeness(count_gdf, size_gdf, alpha=0.1, time_thresh=2, saturation_thresh=1.5, abs_thresh=1.5, return_full=False):
    '''
    Receives two GeoDataFrames (assumes identical timestamps and geometry):
    1) A cumulative count of added features by timestamps
    2) A cumulative value of all features by timestamps

    The function converts timestamp to actual datetime format, transforms the values to a mixed normalized percentage of added value (length / area) per each added unit.
    After that, the function applies the following statistical test:
    
    `If all cumulative change percentage is below some alpha (default: 10%) for a stable time period (default: 2 years) without a large absolute addition (default: 150% more than saturation point), the data is considered saturated.`
    
    For supposedly saturated data, the function computes the saturation point (1st month in stable period) and calculates cumulative percentage up to that point.

    The test is verified using 3 conditions:
    (1) No meaningfull relative addition (percentage<`alpha`) for at least `time_thresh` years.
    (2) No absolute addition (count<`abs_thresh`) for at least `time_thresh` years.
    (3) The absolute or relative addition change percentage since the saturation point is less than `saturation_thresh`.

    In all cases, the output is a DataFrame:
    * If not saturated --> the merged DataFrame, with updated timstamps and calculations.
    * If saturated --> can either return only values up to the saturation point + maximum value (default) or return the entire data with reference to the saturation point.
    
    Dependencies:
    * pandas as pd
    '''
    #---------------------------------------------------#
    #                   Data Wrangle                    #
    #---------------------------------------------------#

    # Fix timestamp
    count_gdf['timestamp'] = pd.to_datetime(count_gdf['timestamp'])
    size_gdf['timestamp'] = pd.to_datetime(size_gdf['timestamp'])

    # Sort both DataFrames by timestamp and reset index for proper alignment
    count_gdf = count_gdf.sort_values('timestamp').reset_index(drop=True)
    size_gdf = size_gdf.sort_values('timestamp').reset_index(drop=True)

    # Merge Dataframes
    gdf = count_gdf.copy().rename(columns={'value' : 'count'}) # Copy DF and rename count column
    gdf['size'] = size_gdf['value'] # Append size column

    #---------------------------------------------------#
    #                 Statistical Test                  #
    #---------------------------------------------------#
    
    # Transform values
    gdf['cumulative_percentage'] = gdf['size'] / gdf['count']
    gdf['cumulative_percentage'] = gdf['cumulative_percentage'].fillna(0) # Deal with periods without addition
    gdf['normalized_cum_per'] = gdf['cumulative_percentage'] / gdf['cumulative_percentage'].max()

    # Adjust alpha value for small to large mapping case
    if gdf['cumulative_percentage'].idxmax() >= (len(gdf) * 0.75):
        alpha = 1 - alpha

    # Apply completeness test for level alpha
    gdf['test'] = (gdf['normalized_cum_per'] < alpha) # Boolean term for each date in data

    #---------------------------------------------------#
    #                   Condititon 1                    #
    #---------------------------------------------------#

    # Iterate backwards in data to find stability period
    i = -1 # Running index (from end)
    test = gdf['test'].iat[i] # Running boolean test answer (from end)
    while test:
        try: # Update index
            i -= 1
            test = gdf['test'].iat[i]
        
        except IndexError: # Break loop if at first index
            break
    
    if i == -1:
        # Deal with last value percentage being greater than alpha (i.e. no stable period)
        return {
            'result': gdf,
            'status': 'incomplete',
            'saturation_point': None,
            'incompletion_reason': 'no stable period'
            }

    stable = gdf.iloc[i+1:].copy() # Extract stable period

    ## If stable period shorter than given time threshold --> data is incomplete
    if (stable['timestamp'].max() - stable['timestamp'].min()) < pd.Timedelta(days=time_thresh*365):
        return {
            'result': gdf,
            'status': 'incomplete',
            'saturation_point': None,
            'incompletion_reason': 'stable period shorter than threshold'
            }
    
    #---------------------------------------------------#
    #             Condititon 3 - 1st phase              #
    #---------------------------------------------------#

    else:
        # Extract saturated value
        saturation_point = stable.iloc[0]

        # Calculate saturation levels
        gdf['percentage_until_saturation'] = gdf['count'] / saturation_point['count']
        
         # Extract emprical maximal value
        real_max = gdf.iloc[-1]
      

        # Check first if data converged. If not, verify if almost converged after one-time event (if exists).
        # Test condition 3 for relative change:
        if (real_max['percentage_until_saturation'] >= saturation_thresh):

    #---------------------------------------------------#
    #                   Condititon 2                    #
    #---------------------------------------------------#

            stable['count_change'] = stable['count'] / stable['count'].max() # Calculate absolute change
            if (stable['count_change'] >= abs_thresh).any():
                # There exists some one-time addition event after saturation
                abs_add_index = stable['count_change'].idxmax()
                if (stable['timestamp'].max() - stable.loc[abs_add_index, 'timestamp']) < pd.Timedelta(days=time_thresh*365):
                    # No stable period since one-time addition event 
                    print('Data incomplete: no stable absolute addition period')
    
    #---------------------------------------------------#
    #             Condititon 3 - 2nd phase              #
    #---------------------------------------------------#

                # Redefine saturation point and levels for one-time addition event
                saturation_point = stable.iloc[abs_add_index]
                gdf['percentage_until_saturation'] = gdf['count'] / saturation_point['count']
                real_max = gdf.iloc[-1]
                
                # Test condition 3 for absoulte change:
                if (real_max['percentage_until_saturation'] >= saturation_thresh):
                    return {
                        'result': gdf,
                        'status': 'incomplete',
                        'saturation_point': None,
                        'incompletion_reason': 'stable absolute addition larger than threshold'
                        }

            
            else:
                return {
                        'result': gdf,
                        'status': 'incomplete',
                        'saturation_point': None,
                        'incompletion_reason': 'stable relative addition larger than threshold'
                        }
    
    #---------------------------------------------------#
    #             Output Saturated Results              #
    #---------------------------------------------------#

        # Extract 80% saturation timestamp
        saturated_time = gdf[gdf['percentage_until_saturation'] >= 0.8]['timestamp'].iloc[0]

        ### Return entire gdf if requested
        if return_full:
            return {
                    'result': gdf,
                    'status': 'complete',
                    'saturation_point': saturated_time,
                    'incompletion_reason': None
                    }
        
        ### Return compact gdf (default)
        else:
            # Filter data until saturation
            saturated = gdf.iloc[:i+1].copy()
        
            # Concatenate empirical maximal value
            saturated = pd.concat([saturated,
                                   pd.DataFrame([real_max])],
                                   ignore_index=True)       
            return {
                    'result': saturated,
                    'status': 'complete',
                    'saturation_point': saturated_time,
                    'incompletion_reason': None
                    }

In [44]:
completeness_region = {}
completeness_region_results = {}
completeness_country = {}
completeness_country_results = {}

for country in df_areas["ISO"].unique():

    area_c = df_areas[df_areas["ISO"] == country]
    count_c = df_counts[df_counts["ISO"] == country]

    # --- inner loop: regions ---
    for region in area_c["region"].unique():

        area_r = area_c[area_c["region"] == region]
        count_r = count_c[count_c["region"] == region]

        if count_r.empty or area_r.empty:
            continue  # 🔴 prevents crash

        region_out = assess_feature_completeness(
            count_gdf=count_r,
            size_gdf=area_r,
        )

        completeness_region[(country, region)] = region_out['result']
        completeness_region_results[(country, region)] = {
            'status': region_out['status'],
            'saturation_point': region_out['saturation_point'],
            'incompletion_reason': region_out['incompletion_reason']
        }

    # --- outer aggregation: country level ---
    country_ts = (
    count_c.groupby("timestamp")["value"].sum()
    .to_frame("count")
    .join(
        area_c.groupby("timestamp")["value"].sum().to_frame("size"),
        how="outer"
    )
    .fillna(0)
    .reset_index()
    .sort_values("timestamp")
    )

    country_out = assess_feature_completeness(
        count_gdf=country_ts.rename(columns={"count": "value"}),
        size_gdf=country_ts.rename(columns={"size": "value"})
        )

    completeness_country[country] = country_out['result']
    completeness_country_results[country] = {
            'status': country_out['status'],
            'saturation_point': country_out['saturation_point'],
            'incompletion_reason': country_out['incompletion_reason']
        }

C:\Users\eliya\AppData\Local\Temp\ipykernel_2028\1231310278.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  count_gdf['timestamp'] = pd.to_datetime(count_gdf['timestamp'])
C:\Users\eliya\AppData\Local\Temp\ipykernel_2028\1231310278.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  size_gdf['timestamp'] = pd.to_datetime(size_gdf['timestamp'])
C:\Users\eliya\AppData\Local\Temp\ipykernel_2028\1231310278.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
T

In [40]:
completeness_region[('EC', 'EC-S')]

,timestamp,count,ISO,region,size,cumulative_percentage,normalized_cum_per,test,percentage_until_saturation
0,2008-01-22 00:00:00+00:00,0.0,EC,EC-S,0.00,0.000000,0.000000,True,0.000000
1,2008-02-22 00:00:00+00:00,0.0,EC,EC-S,0.00,0.000000,0.000000,True,0.000000
2,2008-03-22 00:00:00+00:00,0.0,EC,EC-S,0.00,0.000000,0.000000,True,0.000000
3,2008-04-22 00:00:00+00:00,0.0,EC,EC-S,0.00,0.000000,0.000000,True,0.000000
4,2008-05-22 00:00:00+00:00,0.0,EC,EC-S,0.00,0.000000,0.000000,True,0.000000
...,...,...,...,...,...,...,...,...,...
209,2025-06-22 00:00:00+00:00,31676.0,EC,EC-S,2928134.62,92.440164,0.006204,True,151.559809
210,2025-07-22 00:00:00+00:00,31681.0,EC,EC-S,2929814.56,92.478601,0.006206,True,151.583732
211,2025-08-22 00:00:00+00:00,31687.0,EC,EC-S,2930402.65,92.479649,0.006207,True,151.612440
212,2025-09-22 00:00:00+00:00,31689.0,EC,EC-S,2922824.81,92.234681,0.006190,True,151.622010


In [41]:
completeness_country['EC']

,timestamp,count,size,cumulative_percentage,normalized_cum_per,test,percentage_until_saturation
0,2008-01-22 00:00:00+00:00,0.0,0.000000e+00,0.000000,0.000000,True,0.000000
1,2008-02-22 00:00:00+00:00,0.0,0.000000e+00,0.000000,0.000000,True,0.000000
2,2008-03-22 00:00:00+00:00,0.0,0.000000e+00,0.000000,0.000000,True,0.000000
3,2008-04-22 00:00:00+00:00,0.0,0.000000e+00,0.000000,0.000000,True,0.000000
4,2008-05-22 00:00:00+00:00,0.0,0.000000e+00,0.000000,0.000000,True,0.000000
...,...,...,...,...,...,...,...
209,2025-06-22 00:00:00+00:00,966607.0,1.584038e+08,163.876132,0.021673,True,22.936360
210,2025-07-22 00:00:00+00:00,973587.0,1.593804e+08,163.704354,0.021650,True,23.101986
211,2025-08-22 00:00:00+00:00,984684.0,1.606249e+08,163.123347,0.021574,True,23.365304
212,2025-09-22 00:00:00+00:00,987591.0,1.627796e+08,164.824950,0.021799,True,23.434283


In [49]:
pd.DataFrame(completeness_region_results).T

status saturation_point  \
EC EC-S   incomplete             None   
   EC-Y   incomplete             None   
   EC-N   incomplete             None   
   EC-D   incomplete             None   
   EC-M   incomplete             None   
...              ...              ...   
UY UY-CO  incomplete             None   
   UY-TA  incomplete             None   
   UY-FS  incomplete             None   
   UY-CL  incomplete             None   
   UY-CA  incomplete             None   

                                     incompletion_reason  
EC EC-S   stable relative addition larger than threshold  
   EC-Y                                 no stable period  
   EC-N             stable period shorter than threshold  
   EC-D   stable relative addition larger than threshold  
   EC-M   stable relative addition larger than threshold  
...                                                  ...  
UY UY-CO  stable relative addition larger than threshold  
   UY-TA                                no stable period  
   UY-FS  stable relative addition larger than threshold  
   UY-CL  stable relative addition larger than threshold  
   UY-CA  stable relative addition larger than threshold  

[136 rows x 3 columns]

In [48]:
pd.DataFrame(completeness_country_results).T

,status,saturation_point,incompletion_reason
EC,incomplete,None,stable relative addition larger than threshold
IE,incomplete,None,stable relative addition larger than threshold
IL,incomplete,None,stable relative addition larger than threshold
KR,incomplete,None,stable period shorter than threshold
NP,incomplete,None,stable relative addition larger than threshold
PT,incomplete,None,stable relative addition larger than threshold
SO,incomplete,None,stable relative addition larger than threshold
UY,incomplete,None,stable relative addition larger than threshold


### Results - continue work from here

In [ ]:
comparison = island_data.drop(columns=['geometry', 'bbox', 'instituto_geografico_nacional_espana', 'doi_10_5281_zenodo_8174931', 'usgs_lidar', 'bbox_for_ohsome']).copy()

In [ ]:
# Add total amount of buildings and OSM percentage via overture
comparison['total_bld_overture'] = comparison.drop(columns=['country']).sum(axis=1)
comparison['osm_percentage'] = (comparison['openstreetmap'] / comparison['total_bld_overture']).round(2)

In [ ]:
# Add final parameters extracted
comparison['ohsome_last_count'] = comparison['country'].map(
    {k: v['count'].iloc[-1] for k, v in completeness_dict.items()}
)

comparison['ohsome_last_area'] = comparison['country'].map(
    {k: v['size'].iloc[-1] for k, v in completeness_dict.items()}
)

comparison['last_cum_per'] = comparison['country'].map(
    {k: v['cumulative_percentage'].iloc[-1] for k, v in completeness_dict.items()}
)

comparison['last_norm_cum_per'] = comparison['country'].map(
    {k: v['normalized_cum_per'].iloc[-1] for k, v in completeness_dict.items()}
)

In [ ]:
# Save to csv
comparison.to_csv(r"C:\Users\eliya\Documents\עבודה\עוזר מחקר\4. פרויקטים\1. feature completness\OSM-Feature-Completeness\1) Data\4) overture comparisson\final_island_comparison.csv", index=False)

In [ ]:
# Present the comparison table
from IPython.display import Markdown

Markdown(comparison.to_markdown(index=False))

#### For a level of $\alpha = 0.1$, none of the islands are considered complete:

*All conclusions are based upon the time series data and percentages, as well as the table above*

1) **<u>Iceland (IS)</u>**

**No stable period.**
According to the method, until end date, each month the added cumulative percentage has been larger than $\alpha$, thus resulting in a steady yet constant addition of large buildings to the area. For a level of $\alpha = 0.15$ however, the stable period starts at 2013, but a test would continue yielding none-complete results due to an absolute adition of more than twice data as can be seen in the saturation point (22/09/13).

Further tweaks can be made using the threshold, but according to overture, the data still misses about 30% of buildings (note that Ohsome extracts twice as many buildings that were removed in overture).

2) **<u>Bahrain (BH)</u>**

**Stable relative addition larger than threshold (1.5).**
Although there does seem to be a stable period since 04/2017, there is a constant relative rise in mapping, resulting in an absolute addition of more than 5 times.
Note that in this case, OSM amounts to only 15% of all buildings mapped in Bahrain according to Overture, meaning that we expected the result here to be not complete (the number of objects taken from OSM to Overture seem to be almost identical).

3) **<u>Samoa (WS)</u>**

**Stable relative addition larger than threshold (1.5).**
<u>*This is a problematic case*.</u>

According to Overture, no other sources than OSM exist in this area, yet it still doesn't show up as complete (note that Overture removed about 4,000 buildings). However, if we look at the values themselves, we can see that the time series shows also here a constant rise in data that results in an absolute addition of more than 7 times the amount in the saturation point, which began after the second mapping of buildings in Samoa ever (12/2011). This might show that sensitivity to country size isn't as good as we hoped it to be.

4) **<u>Mauritius (MU)</u>**

**Stable relative addition larger than threshold (1.5).**
Similar to Bahrain, in MU we also see a long stability period since 07/2017 that had a constant small addition, resulting in 8 times more data than in the saturation point. Again, as OSM data amounts to only 14% of all mapped buildings, we could have expected it to come out incomplete.

5) **<u>Fiji (FJ)</u>**

**Stable relative addition larger than threshold (1.5).** This is the strangest case: while Overture used only OSM data here, Ohsome extracted much much more data - about 7 milion buildings, as opposed to 270,000 in Overture. Even stranger, while we see a steady rise since 2010, the relative cumulative percentage keeps getting smaller, yet it doesn't stop at any point, resulting in a crazy absolute addition of more than 1,400 times the saturation point. This can either mean the method doesn't work here, or that Overture isn't more complete.

6) **<u>Tonga (TO)</u>**

**Stable relative addition larger than threshold (1.5).** Here we have another peculiar case: Overture extracted more features than Ohsome did at the endpoint. Like in the case of Fiji, our stable period is very very long, resulting in an absolute addition of 132 times the saturation point.

7) **<u>New Zealand (NZ)</u>**

**Stable relative addition larger than threshold (1.5).** Here, while Ohsome received 5 milion features at the end (unlike the 3 milion overall by Overture), again we see an absolute constant addition that amounts to 100 times the saturation point. Note that in NZ, a one time event (possibly an export) that occured at 03-05/2011 completely obscured the data, adding an extreme area that was later deleted. This resulted in making it the basis of the test and in accordance, an unrealistic value of addition. removing it (as can be seen in the upcoming code chunks) makes the data seem much more sensible (with an absolute addition of only 5 times the saturation, which now occurs at 03/2016), yet still incomplete (as we would have expected without reference to the huge number of features that don't appear in Overture).

8) **<u>Cyprus (CY)</u>**

**Stable relative addition larger than threshold (1.5).** For Cyprus, we see that the saturation point is defined to be 04/2018, and up until that point the percentages behave as we would expect. We can see that there was a large absolute addition - about 200,000 more features over the course of the years, amounting to 6 times as much as the saturated point. We can see that it still misses a lot of data, that can suggest that for us if the data behaves as we expect it to, the method works.

#### Fixing NZ

In [ ]:
fixed_nz_counts = bld_counts[6].copy()
fixed_nz_counts['timestamp'] = pd.to_datetime(fixed_nz_counts['timestamp'], utc=True)
fixed_nz_counts = fixed_nz_counts[
    ~fixed_nz_counts['timestamp'].between(
        pd.Timestamp('2011-03-21', tz='UTC'),
        pd.Timestamp('2011-05-23', tz='UTC')
    )
]

In [ ]:
fixed_nz_areas = bld_areas[6].copy()
fixed_nz_areas['timestamp'] = pd.to_datetime(fixed_nz_areas['timestamp'], utc=True)
fixed_nz_areas = fixed_nz_areas[
    ~fixed_nz_areas['timestamp'].between(
        pd.Timestamp('2011-03-21', tz='UTC'),
        pd.Timestamp('2011-05-23', tz='UTC')
    )
]

In [ ]:
fixed_nz_completeness = assess_feature_completeness(
        count_gdf=fixed_nz_counts,
        size_gdf=fixed_nz_areas
    )

In [ ]:
fixed_nz_completeness

In [ ]:
fixed_nz_completeness.to_csv(r"C:\Users\eliya\Documents\עבודה\עוזר מחקר\4. פרויקטים\1. feature completness\OSM-Feature-Completeness\1) Data\4) overture comparisson\completeness_NZ_fixed.csv")